## Determining the optimal number of hidden layers and neurons

Instead of Experimentation,we can follow the below methods:
<br>
1.Start Simple- start with smaller number and then increase the complexity
<br>
2.Grid Search/Random Search- to try different architectures
<br>
3.Cross-Validation- to evaluate the performance
<br>
4.Heuristic Rules-
<br>
no of neurons in hidden layer is between size of input layer and size of output layer
<br>
A common practice is to start with 1-2 hidden layers



In [1]:
#Modules required
import pandas as pd
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping


In [2]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
#Preprocessing and scaler conversion
data=data.drop(["RowNumber","CustomerId","Surname"],axis=1)
#gender Encode
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data[['Gender']])
data.head()
#geography encode
onehot_encoder_geo=OneHotEncoder()
onehot_encoder_geo_df=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(onehot_encoder_geo_df,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

/home/hp/Documents/GenAI/DeepLearning/lib/python3.8/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [4]:
onehot_encoder_geo.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [ ]:
#create the train and test data and then scale them
X=data.drop('Exited',axis=1)
Y=data['Exited']
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=32)

scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)# we dont fit on the testing data

#no need to save the encoded and scaler files, aswe already did iter

In [31]:
#Function to try out different values for neurons and layers

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(x_train.shape[1],)))
    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [32]:
from scikeras.wrappers import KerasClassifier  #Keras Classifier helps to bridge the gap between keras modela nd sklearn ecosystem
#sklearn helps to do hyper tuning.
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,verbose=1)


In [33]:
#Grid Search parameters for hypertuning
param_grid={
    'neurons':[16,32,64,128],
    'layers':[1,2],
    'epochs':[50,100]
}

In [ ]:
#perform Grid Search CV
from sklearn.model_selection import GridSearchCV
grid=GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=1,cv=3,verbose=1,error_score='raise') 
#n_jobs- to tell number of cpus to use, cv= splitting data into no of sets and then start training
#fit the model
grid_result=grid.fit(x_train,y_train)

##Best parameters after fine tuning
print("Best :%f using %s" % ( grid_result.best_score_,grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Epoch 1/50


/home/hp/Documents/GenAI/DeepLearning/lib/python3.8/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


167/167 [==============================] - 1s 1ms/step - loss: 0.5222 - accuracy: 0.7769
Epoch 2/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4505 - accuracy: 0.7960
Epoch 3/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4284 - accuracy: 0.8048
Epoch 4/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4170 - accuracy: 0.8146
Epoch 5/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4092 - accuracy: 0.8247
Epoch 6/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4027 - accuracy: 0.8305
Epoch 7/50
167/167 [==============================] - 0s 1ms/step - loss: 0.3962 - accuracy: 0.8371
Epoch 8/50
167/167 [==============================] - 0s 1ms/step - loss: 0.3898 - accuracy: 0.8412
Epoch 9/50
167/167 [==============================] - 0s 1ms/step - loss: 0.3828 - accuracy: 0.8451
Epoch 10/50
167/167 [==============================] - 0s 1ms/step - loss: 0.3763 - accuracy: 0.8491
Epoch 11/5

/home/hp/Documents/GenAI/DeepLearning/lib/python3.8/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


167/167 [==============================] - 1s 2ms/step - loss: 0.4967 - accuracy: 0.7769
Epoch 2/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4433 - accuracy: 0.8110
Epoch 3/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4291 - accuracy: 0.8166
Epoch 4/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4207 - accuracy: 0.8215
Epoch 5/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4138 - accuracy: 0.8237
Epoch 6/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4075 - accuracy: 0.8305
Epoch 7/50
167/167 [==============================] - 0s 1ms/step - loss: 0.4007 - accuracy: 0.8329
Epoch 8/50
167/167 [==============================] - 0s 1ms/step - loss: 0.3942 - accuracy: 0.8339
Epoch 9/50
167/167 [==============================] - 0s 1ms/step - loss: 0.3878 - accuracy: 0.8384
Epoch 10/50
167/167 [==============================] - 0s 1ms/step - loss: 0.3814 - accuracy: 0.8412
Epoch 11/5

...

167/167 [==============================] - 5s 1ms/step - loss: 0.4453 - accuracy: 0.8061
Epoch 2/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3678 - accuracy: 0.8500
Epoch 3/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3446 - accuracy: 0.8558
Epoch 4/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3405 - accuracy: 0.8635
Epoch 5/100
167/167 [==============================] - 0s 1ms/step - loss: 0.3324 - accuracy: 0.8584
Epoch 6/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3270 - accuracy: 0.8661
Epoch 7/100
167/167 [==============================] - 0s 1ms/step - loss: 0.3228 - accuracy: 0.8684
Epoch 8/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3185 - accuracy: 0.8659
Epoch 9/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3131 - accuracy: 0.8701
Epoch 10/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3108 - accuracy: 0.8676
E

/home/hp/Documents/GenAI/DeepLearning/lib/python3.8/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


167/167 [==============================] - 1s 2ms/step - loss: 0.4219 - accuracy: 0.8225
Epoch 2/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3547 - accuracy: 0.8547
Epoch 3/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3383 - accuracy: 0.8631
Epoch 4/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3308 - accuracy: 0.8678
Epoch 5/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3220 - accuracy: 0.8682
Epoch 6/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3224 - accuracy: 0.8676
Epoch 7/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3122 - accuracy: 0.8729
Epoch 8/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3142 - accuracy: 0.8721
Epoch 9/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3072 - accuracy: 0.8757
Epoch 10/100
167/167 [==============================] - 0s 2ms/step - loss: 0.3032 - accuracy: 0.8781
E

/home/hp/Documents/GenAI/DeepLearning/lib/python3.8/site-packages/scikeras/wrappers.py:915: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


250/250 [==============================] - 1s 1ms/step - loss: 0.5184 - accuracy: 0.7460
Epoch 2/100
250/250 [==============================] - 0s 1ms/step - loss: 0.4122 - accuracy: 0.8259
Epoch 3/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3883 - accuracy: 0.8401
Epoch 4/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3673 - accuracy: 0.8528
Epoch 5/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3539 - accuracy: 0.8571
Epoch 6/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3473 - accuracy: 0.8608
Epoch 7/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3426 - accuracy: 0.8593
Epoch 8/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3400 - accuracy: 0.8616
Epoch 9/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3376 - accuracy: 0.8621
Epoch 10/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3361 - accuracy: 0.8630
E